# Visualizing pulse schedules across scheduling methods

When you change how pulses are scheduled — a different `timing_policy`, a
different transpiler `scheduling_method`, or a provider code change — you
want to see quickly whether (and where) the resulting Qubex `PulseSchedule`
changed. This tutorial walks through the helpers that make that judgment
easy:

- `build_pulse_schedule_timeline_figure` — Gantt-style timeline, one figure per schedule (like Qiskit's `timeline_drawer`)
- `extract_pulse_timeline` — the same timeline data, programmatically
- `summarize_pulse_schedule` — per-channel timing facts
- `diff_pulse_schedules` — sample-by-sample numerical verdict

Requirements: Qubex (provides the `qxpulse` pulse classes) and the `plot`
extra. For example:

```bash
pip install "qiskit-qubex-provider[qubex,plot] @ git+https://github.com/orangekame3/qiskit-qubex-provider.git"
```

In [1]:
import numpy as np
from qxpulse import Drag, FlatTop, PulseSchedule, VirtualZ

from qiskit_qubex_provider import (
    build_pulse_schedule_timeline_figure,
    diff_pulse_schedules,
    extract_pulse_timeline,
    summarize_pulse_schedule,
    write_pulse_schedule_timeline_image,
)

## On hardware: build schedules without executing

With a configured Qubex `Experiment`, build the schedules for the same
circuit under two scheduling methods via `backend.validate(...)` — no
hardware time is consumed:

```python
from qiskit_qubex_provider import QubexProvider

provider_a = QubexProvider.from_experiment(exp, timing_policy="qiskit")
provider_b = QubexProvider.from_experiment(exp, timing_policy="legacy_device_gateway")

schedule_a = provider_a.get_backend().validate(circuit)[0]
schedule_b = provider_b.get_backend().validate(circuit)[0]
```

Both objects are plain Qubex `PulseSchedule` instances — exactly what we
build by hand below, so the rest of the workflow is identical.

## In this tutorial: hand-built Qubex schedules

We build the same logical content — a drive pulse and a virtual-Z on
`Q00`, a CR-like pulse on `Q01`, a `Drag` correction on `Q00`, and a
readout on `RQ00` — scheduled two different ways. The only difference is a
barrier after the first pulse: variant A waits for it (ALAP-like), variant
B starts the `Q01` pulse immediately (ASAP-like).

In [2]:
def build_demo_schedule(asap: bool) -> PulseSchedule:
    with PulseSchedule(["Q00", "Q01", "RQ00"]) as ps:
        ps.add("Q00", FlatTop(duration=30, amplitude=1.0, tau=10))
        ps.add("Q00", VirtualZ(np.pi / 2))
        if not asap:
            ps.barrier()
        ps.add("Q01", FlatTop(duration=100, amplitude=0.6, tau=10))
        ps.add("Q00", Drag(duration=20, amplitude=0.8, beta=0.1))
        ps.barrier()
        ps.add("RQ00", FlatTop(duration=200, amplitude=0.4, tau=10))
    return ps


schedule_a = build_demo_schedule(asap=False)  # ALAP-like
schedule_b = build_demo_schedule(asap=True)   # ASAP-like

## Gantt-style timeline (Qiskit `timeline_drawer`-like)

`build_pulse_schedule_timeline_figure` renders one horizontal lane per
channel, with a labeled box for every pulse (colored by waveform name) and
a tall labeled tick (e.g. `VZ(-1.57)`) for every virtual-Z phase shift —
the same reading experience as Qiskit's `timeline_drawer` or IQM's playlist
views, but for the Qubex schedule that actually runs. One figure per
schedule: render both variants and compare them side by side.

In [3]:
build_pulse_schedule_timeline_figure(schedule_a, title="ALAP-like").show()

In [4]:
build_pulse_schedule_timeline_figure(schedule_b, title="ASAP-like").show()

The `Q01` pulse and the readout start 30 ns earlier in the ASAP-like
variant — exactly the kind of shift you want to spot when changing the
scheduling method.

To share a figure (e.g. in a PR description), write it to a file.
`.html` needs only Plotly; `.png`/`.svg` additionally need Kaleido.

In [5]:
write_pulse_schedule_timeline_image(schedule_a, "timeline-alap.html", title="ALAP-like")

The timeline data behind the figure is also available programmatically:

In [6]:
extract_pulse_timeline(schedule_a)

{'Q00': [{'kind': 'pulse',
   'name': 'FlatTop',
   'start_ns': 0.0,
   'duration_ns': 30.0},
  {'kind': 'phase',
   'name': 'VirtualZ',
   'start_ns': 30.0,
   'duration_ns': 0.0,
   'theta': -1.5707963267948966},
  {'kind': 'pulse', 'name': 'Drag', 'start_ns': 30.0, 'duration_ns': 20.0}],
 'Q01': [{'kind': 'pulse',
   'name': 'FlatTop',
   'start_ns': 30.0,
   'duration_ns': 100.0}],
 'RQ00': [{'kind': 'pulse',
   'name': 'FlatTop',
   'start_ns': 130.0,
   'duration_ns': 200.0}]}

## Waveform detail via Qubex

For the actual waveform envelopes (X/Y components and frame phase per
channel), call Qubex's own plot on the schedule — no provider API needed:

In [7]:
schedule_a.plot(title="ALAP-like (waveform detail)")

## Per-channel timing summary

`summarize_pulse_schedule` reports, per channel, the total duration and the
first/last non-blank sample boundaries — often enough to spot where a
scheduling change moved a pulse.

In [8]:
summarize_pulse_schedule(schedule_a)

{'Q00': {'duration_ns': 330.0,
  'active_start_ns': 0.0,
  'active_end_ns': 50.0,
  'n_samples': 165},
 'Q01': {'duration_ns': 330.0,
  'active_start_ns': 30.0,
  'active_end_ns': 130.0,
  'n_samples': 165},
 'RQ00': {'duration_ns': 330.0,
  'active_start_ns': 130.0,
  'active_end_ns': 330.0,
  'n_samples': 165}}

In [9]:
summarize_pulse_schedule(schedule_b)

{'Q00': {'duration_ns': 300.0,
  'active_start_ns': 0.0,
  'active_end_ns': 50.0,
  'n_samples': 150},
 'Q01': {'duration_ns': 300.0,
  'active_start_ns': 0.0,
  'active_end_ns': 100.0,
  'n_samples': 150},
 'RQ00': {'duration_ns': 300.0,
  'active_start_ns': 100.0,
  'active_end_ns': 300.0,
  'n_samples': 150}}

## Numerical verdict

`diff_pulse_schedules` compares two schedules sample by sample. Use it as a
quick yes/no answer ("did my refactor change the schedule at all?") or as a
regression check in tests.

Here the barrier change alters each channel's total duration (330 ns vs
300 ns), so every channel reports `length_mismatch`:

In [10]:
diff = diff_pulse_schedules(schedule_a, schedule_b)
print("equal:", diff["equal"])
diff["channels"]

equal: False


{'Q00': {'status': 'length_mismatch',
  'max_abs_diff': None,
  'n_samples_a': 165,
  'n_samples_b': 150},
 'Q01': {'status': 'length_mismatch',
  'max_abs_diff': None,
  'n_samples_a': 165,
  'n_samples_b': 150},
 'RQ00': {'status': 'length_mismatch',
  'max_abs_diff': None,
  'n_samples_a': 165,
  'n_samples_b': 150}}

In [11]:
# Identical schedules compare equal.
diff_pulse_schedules(schedule_a, build_demo_schedule(asap=False))["equal"]

True

## Wrap-up

On hardware, replace the hand-built schedules with the real ones from
`backend.validate(circuit)` and the rest of the workflow is identical.

Related views:

- Waveform detail (X/Y, phase): Qubex's own `schedule.plot()` (shown above)
- Gate-level timeline of the scheduled circuit (`timing_policy="qiskit"`):
  `qiskit.visualization.timeline_drawer(transpiled_circuit)`

See also: [docs/pulse-schedule-visualization.md](../docs/pulse-schedule-visualization.md).